# Pinecone RAG Chunking Experiment — MiniLM

This notebook creates a **standard Pinecone dense-vector index connection** using an index you create yourself in the Pinecone Console.

It uses MODEL_NAME = "llama-text-embed-v2"
 for both document chunks and query embeddings (384 dimensions), then compares six chunking configurations in six namespaces.

## Before running

1. In the Pinecone Console, create a **dense** index:
   - Name: `documents-minilm` (or choose another name and update the notebook below)
   - Dimensions: **384**
   - Metric: **cosine**
   - Cloud/region: choose an option available to your account
2. Have your source PDF ready.
3. Have your 30-question `evaluation_set` ready. You can paste your existing list into the evaluation cell later.

**Important:** This notebook does not create or delete an index. It only connects to the index you create in the Pinecone Console.

## 1. Install required packages

In [1]:
%pip install -q -U pinecone sentence-transformers pypdf langchain-text-splitters tiktoken pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and configuration

In [14]:
import os
import time
import getpass
import numpy as np
import pandas as pd

from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from langchain_text_splitters import CharacterTextSplitter, TokenTextSplitter

INDEX_NAME = "documents-minilm"

# The six chunking configurations use separate namespaces.
CONFIGURATIONS = [
    {"splitter": "character", "chunk_size": 300, "namespace": "character_300"},
    {"splitter": "character", "chunk_size": 500, "namespace": "character_500"},
    {"splitter": "character", "chunk_size": 900, "namespace": "character_900"},
    {"splitter": "token", "chunk_size": 300, "namespace": "token_300"},
    {"splitter": "token", "chunk_size": 500, "namespace": "token_500"},
    {"splitter": "token", "chunk_size": 900, "namespace": "token_900"},
]

OVERLAP = 50
TOP_K = 3
BATCH_SIZE = 100

print("Configuration ready.")

Configuration ready.


## 3. Connect to Pinecone

In [4]:
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_KEY") 

pc = Pinecone(api_key= PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

# Confirm the index exists and inspect its configuration.
description = pc.describe_index(INDEX_NAME)
print("Index:", description.name)
print("Dimension:", description.dimension)
print("Metric:", description.metric)

if description.dimension != 384:
    raise ValueError(
        f"This notebook uses MiniLM (384 dimensions), but the index has "
        f"{description.dimension} dimensions. Create/use a 384-dimensional index."
    )

if description.metric != "cosine":
    print("WARNING: This notebook expects a cosine-metric index.")

Index: documents-minilm
Dimension: 384
Metric: cosine


## 4. Load MiniLM embedding model

In [5]:
description = pc.describe_index(INDEX_NAME)

print(description)

IndexModel(name='documents-minilm', status='Ready', host='https://documents-minilm-18eadba.svc.aped-4627-b74a.pinecone.io', deployment='Managed', deletion_protection='disabled', schema_fields=2)


In [13]:
MODEL_NAME = "llama-text-embed-v2"
EMBEDDING_DIMENSION = 384
embedding_model = SentenceTransformer(MODEL_NAME)

EMBEDDING_DIMENSION = embedding_model.get_embedding_dimension()
print("Model:", MODEL_NAME)
print("Embedding dimensions:", EMBEDDING_DIMENSION)

assert EMBEDDING_DIMENSION == 384

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6afbde7f-0f191a9e2417eeae64bdf56c;b05bfbaa-d412-46e0-a886-44758830c7af)

Repository Not Found for url: https://huggingface.co/api/models/sentence-transformers/llama-text-embed-v2.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

## 5. Load your PDF

Set `PDF_PATH` to the location of your PDF file. For example, on Windows:

`PDF_PATH = r"C:\Users\YourName\Downloads\python_tutorial.pdf"`

If you already have the extracted `document_text` variable from your previous notebook, you can skip the PDF-reading cell and use that variable.

In [6]:
PDF_PATH = "uploads/Python Tutorial.pdf"

reader = PdfReader(PDF_PATH)
pages = [page.extract_text() or "" for page in reader.pages]
document_text = "\\n".join(pages)

print("Pages:", len(reader.pages))
print("Extracted characters:", len(document_text))
print(document_text[:1000])

Pages: 155
Extracted characters: 309085
Python Tutorial
Release 3.7.0
Guido van Rossum
and the Python development team
September 02, 2018
Python Software Foundation
Email: docs@python.org\n\nCONTENTS
1 Whetting Your Appetite 3
2 Using the Python Interpreter 5
2.1 Invoking the Interpreter . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5
2.2 The Interpreter and Its Environment . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 6
3 An Informal Introduction to Python 9
3.1 Using Python as a Calculator . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 9
3.2 First Steps Towards Programming . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 16
4 More Control Flow Tools 19
4.1 if Statements . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 19
4.2 for Statements . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 19
4.3 The range() Function . .

## 6. Create the six chunk sets

In [7]:
def create_chunks(document_text, method, chunk_size, chunk_overlap=50):
    if method == "character":
        splitter = CharacterTextSplitter(
            separator="\\n",
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            is_separator_regex=False
        )
    elif method == "token":
        splitter = TokenTextSplitter(
            encoding_name="cl100k_base",
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )
    else:
        raise ValueError("method must be 'character' or 'token'")

    return splitter.split_text(document_text)


chunk_sets = {}

for cfg in CONFIGURATIONS:
    namespace = cfg["namespace"]
    chunks = create_chunks(
        document_text,
        method=cfg["splitter"],
        chunk_size=cfg["chunk_size"],
        chunk_overlap=OVERLAP
    )
    chunk_sets[namespace] = chunks
    print(f"{namespace}: {len(chunks)} chunks")

chunk_counts_df = pd.DataFrame([
    {
        "Namespace": cfg["namespace"],
        "Method": cfg["splitter"],
        "Chunk Size": cfg["chunk_size"],
        "Unit": "characters" if cfg["splitter"] == "character" else "tokens",
        "Overlap": OVERLAP,
        "Total Chunks": len(chunk_sets[cfg["namespace"]])
    }
    for cfg in CONFIGURATIONS
])

display(chunk_counts_df)

Created a chunk of size 3020, which is longer than the specified 300
Created a chunk of size 4785, which is longer than the specified 300
Created a chunk of size 813, which is longer than the specified 300
Created a chunk of size 1982, which is longer than the specified 300
Created a chunk of size 3363, which is longer than the specified 300
Created a chunk of size 1091, which is longer than the specified 300
Created a chunk of size 2850, which is longer than the specified 300
Created a chunk of size 2546, which is longer than the specified 300
Created a chunk of size 455, which is longer than the specified 300
Created a chunk of size 1882, which is longer than the specified 300
Created a chunk of size 1901, which is longer than the specified 300
Created a chunk of size 1445, which is longer than the specified 300
Created a chunk of size 304, which is longer than the specified 300
Created a chunk of size 1585, which is longer than the specified 300
Created a chunk of size 2047, which i

character_300: 172 chunks
character_500: 166 chunks


Created a chunk of size 1921, which is longer than the specified 900
Created a chunk of size 1941, which is longer than the specified 900
Created a chunk of size 2168, which is longer than the specified 900
Created a chunk of size 3139, which is longer than the specified 900
Created a chunk of size 4285, which is longer than the specified 900
Created a chunk of size 2507, which is longer than the specified 900
Created a chunk of size 2425, which is longer than the specified 900
Created a chunk of size 2703, which is longer than the specified 900
Created a chunk of size 2406, which is longer than the specified 900
Created a chunk of size 2124, which is longer than the specified 900
Created a chunk of size 2569, which is longer than the specified 900
Created a chunk of size 2693, which is longer than the specified 900
Created a chunk of size 3437, which is longer than the specified 900
Created a chunk of size 2239, which is longer than the specified 900
Created a chunk of size 1417, whic

character_900: 157 chunks
token_300: 309 chunks
token_500: 172 chunks
token_900: 91 chunks


,Namespace,Method,Chunk Size,Unit,Overlap,Total Chunks
0,character_300,character,300,characters,50,172
1,character_500,character,500,characters,50,166
2,character_900,character,900,characters,50,157
3,token_300,token,300,tokens,50,309
4,token_500,token,500,tokens,50,172
5,token_900,token,900,tokens,50,91


## 7. Embed and upload chunks to Pinecone

This uses MiniLM to create **384-dimensional vectors** for every chunk. The text is stored in vector metadata so it can be displayed when searching.

Run this cell once for a fresh set of namespaces. If you run it again after changing the PDF or chunking settings, old vectors may remain in the namespaces. For a clean re-upload, delete the relevant namespace vectors in the Pinecone Console first, or use a new index.

In [8]:
description = pc.describe_index(INDEX_NAME)

print(description.schema)

IndexSchema(fields={'title': StringField(description=None, filterable=False, full_text_search=FullTextSearchConfig(language='en', stemming=False, stop_words=False, ngram=None), embed=None, sparse_embed=None, store_text=True), 'chunk_text': StringField(description=None, filterable=False, full_text_search=FullTextSearchConfig(language='en', stemming=False, stop_words=False, ngram=None), embed=FieldEmbedConfig(model='llama-text-embed-v2', dimension=384, metric='cosine', processing_location=None, write_parameters={'truncate': 'END', 'input_type': 'passage'}, read_parameters={'truncate': 'END', 'input_type': 'query'}), sparse_embed=None, store_text=True)})


In [11]:
for cfg in CONFIGURATIONS:
    namespace = cfg["namespace"]
    chunks = chunk_sets[namespace]

    print(f"Embedding and uploading {namespace} ({len(chunks)} chunks)...")

    documents = []

    for chunk_number, chunk in enumerate(chunks):
        documents.append({
            "_id": f"{namespace}_{chunk_number}",
            "chunk_text": chunk,
            "title": f"{namespace}_chunk_{chunk_number}"
        })

    for start in range(0, len(documents), 90):
        batch = documents[start:start + 90]

        index.documents.upsert(
            namespace=namespace,
            documents=batch
        )

    print(f"Uploaded {len(documents)} documents to {namespace}")

print("Upload completed.")

Embedding and uploading character_300 (172 chunks)...
Uploaded 172 documents to character_300
Embedding and uploading character_500 (166 chunks)...
Uploaded 166 documents to character_500
Embedding and uploading character_900 (157 chunks)...
Uploaded 157 documents to character_900
Embedding and uploading token_300 (309 chunks)...
Uploaded 309 documents to token_300
Embedding and uploading token_500 (172 chunks)...
Uploaded 172 documents to token_500
Embedding and uploading token_900 (91 chunks)...
Uploaded 91 documents to token_900
Upload completed.


## 8. Verify namespace counts

In [12]:
stats = index.describe_index_stats()
namespace_stats = stats.namespaces or {}

verification_rows = []
for cfg in CONFIGURATIONS:
    ns = cfg["namespace"]
    ns_info = namespace_stats.get(ns)
    vector_count = ns_info.vector_count if ns_info else 0
    verification_rows.append({
        "Namespace": ns,
        "Expected Chunks": len(chunk_sets[ns]),
        "Pinecone Vector Count": vector_count,
        "Counts Match": vector_count == len(chunk_sets[ns])
    })

display(pd.DataFrame(verification_rows))

,Namespace,Expected Chunks,Pinecone Vector Count,Counts Match
0,character_300,172,172,True
1,character_500,166,166,True
2,character_900,157,157,True
3,token_300,309,309,True
4,token_500,172,172,True
5,token_900,91,91,True


## 9. Test a query using a MiniLM query embedding

In [18]:
def search_pinecone(question, namespace, top_k=3):

    results = index.documents.search(
        namespace=namespace,
        top_k=top_k,
        score_by=[
            {
                "type": "embed",
                "fields": ["chunk_text"],
                "query": question
            }
        ],
        include_fields=["title", "chunk_text"]
    )

    return results

In [17]:
import inspect

print(inspect.signature(index.documents.search))

(*, namespace: 'str', score_by: 'Sequence[DocumentScoringMethod | Mapping[str, Any]]', top_k: 'int', include_fields: 'Sequence[str] | None' = None, filter: 'Mapping[str, Any] | None' = None, timeout: 'float | None' = None) -> 'SearchDocumentsResponse'


In [20]:
print(test_results)
print(type(test_results))
print(dir(test_results))

SearchDocumentsResponse(matches=3, namespace='token_300', usage=DocumentSearchUsage(read_units=1, embed_total_tokens=7))
<class 'pinecone.models.documents.responses.SearchDocumentsResponse'>
['__annotations__', '__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__static_attributes__', '__str__', '__subclasshook__', '_repr_html_', 'from_dict', 'matches', 'namespace', 'response_info', 'to_dict', 'usage']


In [23]:
test_question = "What is Python?"
test_namespace = "token_300"

test_results = search_pinecone(
    test_question,
    test_namespace,
    top_k=3
)

for rank, match in enumerate(test_results.matches, start=1):
    print(f"Rank: {rank}")
    print("Document:", match)
    print("-" * 80)

Rank: 1
Document: Document(_id='token_300_18', score=0.4532328, ...)
--------------------------------------------------------------------------------
Rank: 2
Document: Document(_id='token_300_19', score=0.4448847, ...)
--------------------------------------------------------------------------------
Rank: 3
Document: Document(_id='token_300_20', score=0.43410927, ...)
--------------------------------------------------------------------------------


## 10. Add your evaluation questions

Paste your existing 30-question `evaluation_set` here. Keep the same structure, including `question` and `expected_answer`.

Example:

```python
evaluation_set = [
    {
        "question": "What is Python?",
        "expected_answer": "Python is an easy to learn and powerful programming language."
    },
    {
        "question": "Why is Python useful?",
        "expected_answer": "Python helps automate tasks and rapid application development."
    }
]
```

The semantic evaluation below does not use keywords. It compares each retrieved chunk with the expected answer using MiniLM embeddings.

In [24]:
evaluation_set = [
    {
        "question": "What is Python?",
        "expected_answer": "Python is an easy to learn and powerful programming language.",
        "keywords": ["easy to learn", "powerful", "programming language"]
    },
    {
        "question": "Why is Python useful?",
        "expected_answer": "Python helps automate tasks and rapid application development.",
        "keywords": ["automate", "rapid application development"]
    },
    {
        "question": "What are the advantages of Python?",
        "expected_answer": "Python offers high-level data structures, object-oriented programming, and interpreted execution.",
        "keywords": ["high-level", "object-oriented", "interpreted"]
    },
    {
        "question": "What is the Python interpreter?",
        "expected_answer": "The interpreter executes Python code interactively or from scripts.",
        "keywords": ["interpreter", "execute", "scripts"]
    },
    {
        "question": "What is interactive mode?",
        "expected_answer": "Interactive mode allows commands to be executed immediately from the prompt.",
        "keywords": ["interactive", "prompt", "commands"]
    },
    {
        "question": "What encoding is used by default in Python source files?",
        "expected_answer": "UTF-8.",
        "keywords": ["UTF-8"]
    },
    {
        "question": "What is floor division?",
        "expected_answer": "Floor division uses // and discards the fractional part.",
        "keywords": ["//", "fractional part"]
    },
    {
        "question": "How do you calculate powers in Python?",
        "expected_answer": "The ** operator is used for exponentiation.",
        "keywords": ["**", "power"]
    },
    {
        "question": "What error occurs when an undefined variable is used?",
        "expected_answer": "NameError.",
        "keywords": ["NameError"]
    },
    {
        "question": "How are strings created in Python?",
        "expected_answer": "Strings can be enclosed in single or double quotes.",
        "keywords": ["single quotes", "double quotes"]
    },
    {
        "question": "What is string slicing?",
        "expected_answer": "Slicing extracts a portion of a string using index ranges.",
        "keywords": ["slice", "substring"]
    },
    {
        "question": "Why are strings immutable?",
        "expected_answer": "String contents cannot be modified after creation.",
        "keywords": ["immutable", "cannot be changed"]
    },
    {
        "question": "What is a list in Python?",
        "expected_answer": "A mutable sequence used to store collections of values.",
        "keywords": ["mutable", "sequence"]
    },
    {
        "question": "What does append() do?",
        "expected_answer": "Adds an element to the end of a list.",
        "keywords": ["append", "end of list"]
    },
    {
        "question": "What is the difference between mutable and immutable objects?",
        "expected_answer": "Mutable objects can be modified while immutable objects cannot.",
        "keywords": ["mutable", "immutable"]
    },
    {
        "question": "How does a while loop work?",
        "expected_answer": "It executes while a condition remains true.",
        "keywords": ["while", "condition"]
    },
    {
        "question": "What is an if statement used for?",
        "expected_answer": "Conditional execution of code.",
        "keywords": ["condition", "branching"]
    },
    {
        "question": "How does a for loop work in Python?",
        "expected_answer": "It iterates over items in a sequence.",
        "keywords": ["iterate", "sequence"]
    },
    {
        "question": "What is the purpose of range()?",
        "expected_answer": "Generate a sequence of numbers.",
        "keywords": ["range", "sequence of numbers"]
    },
    {
        "question": "What does break do?",
        "expected_answer": "Terminates the nearest loop.",
        "keywords": ["break", "loop"]
    },
    {
        "question": "What does continue do?",
        "expected_answer": "Skips the remaining code and starts the next iteration.",
        "keywords": ["continue", "next iteration"]
    },
    {
        "question": "What is the pass statement?",
        "expected_answer": "A placeholder statement that performs no action.",
        "keywords": ["pass", "placeholder"]
    },
    {
        "question": "What is a function?",
        "expected_answer": "A reusable block of code defined using def.",
        "keywords": ["function", "def"]
    },
    {
        "question": "What is a docstring?",
        "expected_answer": "A string used to document a function, class, or module.",
        "keywords": ["docstring", "documentation"]
    },
    {
        "question": "What are default arguments?",
        "expected_answer": "Arguments that have predefined values.",
        "keywords": ["default argument"]
    },
    {
        "question": "What are keyword arguments?",
        "expected_answer": "Arguments passed using parameter names.",
        "keywords": ["keyword argument"]
    },
    {
        "question": "What is a lambda function?",
        "expected_answer": "A small anonymous function created with lambda.",
        "keywords": ["lambda", "anonymous function"]
    },
    {
        "question": "What is a tuple?",
        "expected_answer": "An immutable sequence of values.",
        "keywords": ["tuple", "immutable"]
    },
    {
        "question": "What is a set?",
        "expected_answer": "An unordered collection of unique elements.",
        "keywords": ["set", "unique"]
    },
    {
        "question": "What is a dictionary?",
        "expected_answer": "A collection of key-value pairs.",
        "keywords": ["dictionary", "key", "value"]
    }
]



## 11. Evaluate retrieval and semantic relevance

In [26]:
def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1, dtype=np.float32)
    vec2 = np.asarray(vec2, dtype=np.float32)
    denominator = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if denominator == 0:
        return 0.0
    return float(np.dot(vec1, vec2) / denominator)

def evaluate_namespace(namespace, evaluation_set, top_k=3):
    question_rows = []
    retrieval_times = []

    for item in evaluation_set:
        question = item["question"]
        expected_answer = item["expected_answer"]

        start_time = time.perf_counter()
        results = search_pinecone(question, namespace, top_k=top_k)
        elapsed = time.perf_counter() - start_time
        retrieval_times.append(elapsed)

        matches = results.matches

        # Extract chunk text from Pinecone documents
        chunks = [
            match.chunk_text
            for match in matches
            if getattr(match, "chunk_text", None)
        ]

        if not chunks:
            question_rows.append({
                "namespace": namespace,
                "question": question,
                "expected_answer": expected_answer,
                "best_semantic_similarity": 0.0,
                "average_semantic_similarity": 0.0,
                "retrieval_time_sec": round(elapsed, 4),
                "retrieved_chunks": []
            })
            continue

        # Use the same Llama embedding model for expected answer and chunks
        texts_to_encode = [expected_answer] + chunks

        vectors = encode_texts(texts_to_encode, input_type="passage")

        expected_vector = vectors[0]
        chunk_vectors = vectors[1:]

        semantic_scores = [
            cosine_similarity(expected_vector, chunk_vector)
            for chunk_vector in chunk_vectors
        ]

        question_rows.append({
            "namespace": namespace,
            "question": question,
            "best_semantic_similarity": round(max(semantic_scores), 4),
            "average_semantic_similarity": round(float(np.mean(semantic_scores)), 4),
            "retrieval_time_sec": round(elapsed, 4),
            "retrieved_chunks": chunks
        })

    return question_rows

In [30]:
MODEL_NAME = "llama-text-embed-v2"
EMBEDDING_DIMENSION = 384

def encode_texts(texts, input_type="passage"):
    response = pc.inference.embed(
        model=MODEL_NAME,
        inputs=texts,
        parameters={
            "input_type": input_type,
            "dimension": EMBEDDING_DIMENSION
        }
    )

    return [item.values for item in response]

In [31]:
all_question_details = []

for cfg in CONFIGURATIONS:
    namespace = cfg["namespace"]

    print(f"Evaluating {namespace}...")

    details = evaluate_namespace(
        namespace,
        evaluation_set,
        top_k=TOP_K
    )

    all_question_details.extend(details)

pinecone_question_details_df = pd.DataFrame(all_question_details)

print("Evaluation completed!")
print("Total rows:", len(pinecone_question_details_df))

display(pinecone_question_details_df.head())

Evaluating character_300...
Evaluating character_500...
Evaluating character_900...
Evaluating token_300...
Evaluating token_500...
Evaluating token_900...
Evaluation completed!
Total rows: 180


,namespace,question,best_semantic_similarity,average_semantic_similarity,retrieval_time_sec,retrieved_chunks
0,character_300,What is Python?,0.5250,0.4246,1.4981,"[Python Tutorial, Release 3.7.0\nspecial metho..."
1,character_300,Why is Python useful?,0.4965,0.4533,0.4385,[CHAPTER\nONE\nWHETTING YOUR APPETITE\nIf you ...
2,character_300,What are the advantages of Python?,0.5594,0.5044,0.3487,[CHAPTER\nONE\nWHETTING YOUR APPETITE\nIf you ...
3,character_300,What is the Python interpreter?,0.6202,0.5743,0.3437,[CHAPTER\nTW\nO\nUSING\nTHE PYTHON INTERPRETER...
4,character_300,What is interactive mode?,0.4944,0.4469,0.2647,[CHAPTER\nSIXTEEN\nAPPENDIX\n16.1 Interactive ...


## 12. Compare the six configurations

In [32]:
summary_rows = []

for cfg in CONFIGURATIONS:
    namespace = cfg["namespace"]

    rows = pinecone_question_details_df[
        pinecone_question_details_df["namespace"] == namespace
    ]

    summary_rows.append({
        "Method": cfg["splitter"].capitalize(),
        "Chunk Size": cfg["chunk_size"],
        "Unit": "characters" if cfg["splitter"] == "character" else "tokens",
        "Overlap": OVERLAP,
        "Total Chunks": len(chunk_sets[namespace]),

        "Avg Best Semantic Relevance": round(
            rows["best_semantic_similarity"].mean(), 4
        ) if len(rows) else 0,

        "Avg Top-3 Semantic Relevance": round(
            rows["average_semantic_similarity"].mean(), 4
        ) if len(rows) else 0,

        "Avg Retrieval Time (sec)": round(
            rows["retrieval_time_sec"].mean(), 4
        ) if len(rows) else 0
    })

pinecone_results_df = pd.DataFrame(summary_rows).sort_values(
    "Avg Best Semantic Relevance",
    ascending=False
).reset_index(drop=True)

display(pinecone_results_df)

,Method,Chunk Size,Unit,Overlap,Total Chunks,Avg Best Semantic Relevance,Avg Top-3 Semantic Relevance,Avg Retrieval Time (sec)
0,Token,300,tokens,50,309,0.4202,0.3670,0.3356
1,Token,500,tokens,50,172,0.4073,0.3430,0.2973
2,Character,300,characters,50,172,0.4003,0.3492,0.4019
3,Character,500,characters,50,166,0.3989,0.3481,0.3191
4,Character,900,characters,50,157,0.3980,0.3475,0.3514
5,Token,900,tokens,50,91,0.3782,0.3307,0.3110


## 13. Inspect retrieved chunks for a question

Use this to check whether high semantic scores actually correspond to useful context. A similarity score is a ranking signal, not proof that the chunk answers the question.

In [34]:
question_to_inspect = "What is Python?"
namespace_to_inspect = "token_300"

inspection = pinecone_question_details_df[
    (pinecone_question_details_df["namespace"] == namespace_to_inspect) &
    (pinecone_question_details_df["question"] == question_to_inspect)
]

if inspection.empty:
    print("No matching evaluation row. Check the question and namespace.")
else:
    row = inspection.iloc[0]
    print("Question:", row["question"])
    print("Best semantic similarity:", row["best_semantic_similarity"])
    for i, chunk in enumerate(row["retrieved_chunks"], start=1):
        print(f"\\n--- Retrieved chunk {i} ---\\n{chunk}")

Question: What is Python?
Best semantic similarity: 0.5623
\n--- Retrieved chunk 1 ---\n take a lot of development time to get even a ﬁrst-draft program.
Python is simpler to use, available on Windows, Mac OS X, and Unix operating systems, and will help you
get the job done more quickly.
Python is simple to use, but it is a real programming language, oﬀering much more structure and support
for large programs than shell scripts or batch ﬁles can oﬀer. On the other hand, Python also oﬀers much
more error checking than C, and, being a very-high-level language, it has high-level data types built in, such
as ﬂexible arrays and dictionaries. Because of its more general data types Python is applicable to a much
larger problem domain than Awk or even Perl, yet many things are at least as easy in Python as in those
languages.
Python allows you to split your program into modules that can be reused in other Python programs. It
comes with a large collection of standard modules that you can use as 

In [35]:
comparison_df = pinecone_results_df.copy()

# Normalize relevance: higher is better
comparison_df["Best_Relevance_Normalized"] = (
    comparison_df["Avg Best Semantic Relevance"] /
    comparison_df["Avg Best Semantic Relevance"].max()
)

comparison_df["Top3_Relevance_Normalized"] = (
    comparison_df["Avg Top-3 Semantic Relevance"] /
    comparison_df["Avg Top-3 Semantic Relevance"].max()
)

# Normalize retrieval time: lower is better
comparison_df["Speed_Normalized"] = (
    comparison_df["Avg Retrieval Time (sec)"].min() /
    comparison_df["Avg Retrieval Time (sec)"]
)

# Weighted score: prioritize relevance over speed
comparison_df["Overall Score"] = (
    0.4 * comparison_df["Best_Relevance_Normalized"] +
    0.4 * comparison_df["Top3_Relevance_Normalized"] +
    0.2 * comparison_df["Speed_Normalized"]
)

comparison_df = comparison_df.sort_values(
    "Overall Score", ascending=False
).reset_index(drop=True)

display(comparison_df[
    [
        "Method",
        "Chunk Size",
        "Avg Best Semantic Relevance",
        "Avg Top-3 Semantic Relevance",
        "Avg Retrieval Time (sec)",
        "Overall Score"
    ]
])

,Method,Chunk Size,Avg Best Semantic Relevance,Avg Top-3 Semantic Relevance,Avg Retrieval Time (sec),Overall Score
0,Token,300,0.4202,0.3670,0.3356,0.977175
1,Token,500,0.4073,0.3430,0.2973,0.961562
2,Character,500,0.3989,0.3481,0.3191,0.945461
3,Character,900,0.3980,0.3475,0.3514,0.926823
4,Token,900,0.3782,0.3307,0.3110,0.911645
5,Character,300,0.4003,0.3492,0.4019,0.909603


## Notes

- Query and document vectors must use the **same embedding model**. This notebook uses MiniLM for both.
- The Pinecone index must be **384 dimensions** with **cosine** metric.
- All six configurations are stored in separate namespaces in the same index.
- `match.score` is Pinecone's cosine similarity for this index. The separate semantic relevance score compares the retrieved chunk against the expected answer using MiniLM.
- Do not compare raw similarity values across different embedding models.
- The notebook measures retrieval time for the Pinecone search call; it does not include PDF extraction or document embedding/upload time.
- For a fair ChromaDB vs Pinecone comparison, use identical chunk text, overlap, model, query set, and `top_k`.